# POLSCI 3

## Week 6, Section Activity: Is There a Home-Court Advantage in the WNBA?

**What you'll practice:** quantifying uncertainty. In lecture you learned that an estimate can be wrong for two very different reasons: **bias** and **noise**. Today you'll use real WNBA games to see the difference.

- **Part 1 (everyone):** uses only tools you've already seen in this class.
- **Part 2 (optional):** teaches new R code that goes beyond what we cover in class. Only start it if you finish Part 1.

**Heads up:** when you run `difference_in_means()` you'll see some extra columns (`Pr(>|t|)`, `CI Lower`, `CI Upper`, `DF`). We'll learn those in upcoming lectures, so ignore them for today. Focus on **Estimate**, **Std. Error**, and **t value**.

For written questions, double-click the answer cell, type your answer, and press Shift + Enter.

---
# PART 1: Is the home-court advantage real, or just noise?

### Setup

In [ ]:
library(estimatr)
library(ggplot2)

wnba <- read.csv('ps3_wnba_teamgames.csv')
head(wnba)

### Codebook

Each row is **one team in one game** (so every game shows up twice: once for the home team and once for the away team). The data covers regular-season games from 2022 to 2025.

- `season`: year of the season
- `game_id`: identifier for the game
- `game_date`: date of the game
- `team`: team name
- `home_away`: `"home"` if the team played in its own arena, `"away"` if not
- `points`: points the team scored in that game
- `opp_points`: points the other team scored in that game
- `win`: 1 if the team won, 0 if it lost
- `assists`, `rebounds`, `turnovers`, `threes_made`: other team stats from the game

**Our question:** does playing at home cause a team to score more points?

## Step 1: Understand the data

In [ ]:
# How many rows are there in each season?
table(wnba$season)

# How many rows are "home" versus "away"?
table(wnba$home_away)

**Q1a.** Why are there exactly the same number of "home" rows as "away" rows? (Hint: reread the codebook.)

**Your answer:** *(double-click here to type your answer)*

**Q1b.** In this question, what is the **treatment** and what is the **outcome**? Write the hypothesis in one sentence.

**Your answer:** *(double-click here to type your answer)*

**Q1c.** Is this an experiment? Who (or what) decided which team was at home in each game? Based on Weeks 3-5, what should we be worried about?

**Your answer:** *(double-click here to type your answer)*

## Step 2: Estimate the home-court gap in one season

We'll start with just the 2023 season. Run this cell to create the 2023 data.

In [ ]:
wnba.2023 <- subset(wnba, season == 2023)
head(wnba.2023)

**Q2a. The "by hand" way.** Make one subset with the home rows and one with the away rows, find the mean points in each, and subtract (home minus away).

*Hint: `subset(dataset, variable == 'value')`. Text values need quotes.*

In [ ]:
home.games <- NULL # YOUR CODE HERE
away.games <- NULL # YOUR CODE HERE

home.mean <- NULL # YOUR CODE HERE (mean points in home.games)
away.mean <- NULL # YOUR CODE HERE (mean points in away.games)

home.mean - away.mean

**Q2b. The `difference_in_means()` way.** Estimate the same thing with `difference_in_means()`, using `'away'` as the baseline (`condition1`) and `'home'` as the treatment (`condition2`). Save it in `dim.2023`.

*Hint: `difference_in_means(outcome ~ treatment, data = dataset, condition1 = 'baseline', condition2 = 'treatment')`*

In [ ]:
dim.2023 <- NULL # YOUR CODE HERE
dim.2023

**Q2c.** Does the **Estimate** match your by-hand answer from Q2a? What would happen to the estimate if you swapped `condition1` and `condition2`?

**Your answer:** *(double-click here to type your answer)*

**Q2d.** Write one sentence interpreting the estimate, with units.

**Your answer:** *(double-click here to type your answer)*

## Step 3: Could that gap just be noise?

Remember from lecture: even if home/away did **nothing**, the estimate wouldn't be exactly 0. Chance alone (which games happened to be home games, who got hot) adds **noise**. The **standard error** measures how much noise there is, and the **t-statistic** ($\frac{\text{Estimate}}{\text{Standard Error}}$) tells us how big the estimate is compared with that noise.

**Q3a.** Copy the **Estimate** and **Std. Error** from your `dim.2023` output into the variables below (type the numbers), then compute the t-statistic.

In [ ]:
estimate <- NULL   # type the Estimate from your output
std.error <- NULL  # type the Std. Error from your output

t.stat <- estimate / std.error
t.stat

**Q3b.** Compare your t-statistic to the 1.96 convention from lecture. Is the 2023 home-court gap clearly distinguishable from noise? Answer in one or two sentences.

**Your answer:** *(double-click here to type your answer)*

**Q3c.** In your own words, what does the standard error measure here? Is it about bias or noise?

**Your answer:** *(double-click here to type your answer)*

### Seeing the noise

To see what chance alone can do, imagine a skeptic who says *"home/away makes no difference."* In the skeptic's world, it doesn't matter which rows are labeled home or away, so we can **shuffle the labels** and see how big a gap we get by luck. The code below does this 1,000 times. **You don't need to write it, just run it.**

(`set.seed()` just makes the random shuffles come out the same each time.)

In [ ]:
re.randomize <- function(input.data) {
    input.data$home_away <- sample(input.data$home_away)
    return(input.data)
}

compute.home.effect <- function(input.data) {
    result <- difference_in_means(points ~ home_away, data = input.data,
                                  condition1 = 'away', condition2 = 'home')
    return(result$coefficients)
}

actual.estimate <- compute.home.effect(wnba.2023)

set.seed(3)
simulations <- replicate(1000, compute.home.effect(re.randomize(wnba.2023)))

qplot(simulations, binwidth = .25) + geom_vline(xintercept = actual.estimate, color = 'red')

**Q3d.** The histogram shows the home-minus-away gaps we get from shuffling alone. The red line is our actual 2023 estimate. Where does the red line fall? Do shuffled gaps this big happen rarely, sometimes, or often?

**Your answer:** *(double-click here to type your answer)*

In [ ]:
sd(simulations)

**Q3e.** Compare `sd(simulations)` to the Std. Error in your `dim.2023` output. What do you notice, and why does that make sense?

**Your answer:** *(double-click here to type your answer)*

## Step 4: Where does the standard error come from?

From lecture, for comparing two groups:

$$\text{SE} = \sqrt{\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}}$$

where $s$ is the standard deviation of the outcome in each group and $n$ is the number of observations in each group. (In R, `x^0.5` takes a square root.)

**Q4a.** Compute the standard error by hand for 2023.

In [ ]:
table(wnba.2023$home_away)  # group sizes

s.home <- NULL # standard deviation of points in home.games (hint: sd())
s.away <- NULL # standard deviation of points in away.games
n.home <- NULL # number of home rows (type it in from the table above)
n.away <- NULL # number of away rows

se.by.hand <- NULL # (s.home^2 / n.home + s.away^2 / n.away)^0.5
se.by.hand

**Q4b.** Does your by-hand SE match the Std. Error from `difference_in_means()`?

**Your answer:** *(double-click here to type your answer)*

**Q4c. What if we had more data?** Suppose the league played **4 times as many games** but the standard deviations stayed the same. Recompute the SE using `4 * n.home` and `4 * n.away`.

In [ ]:
se.quadruple <- NULL # YOUR CODE HERE
se.quadruple

se.quadruple / se.by.hand

**Q4d.** By what factor did the SE shrink when the sample grew 4x? How much bigger would the sample need to be to cut the SE to **one third** of its original size?

**Your answer:** *(double-click here to type your answer)*

**Q4e. A design question.** Imagine instead an *experiment* with 482 people in total, where the outcome has a standard deviation of 11 in both groups. Compute the SE for an even split (241 and 241) and for a lopsided split (434 and 48). Which is more precise?

In [ ]:
s <- 11

se.even <- NULL     # YOUR CODE HERE (241 and 241)
se.lopsided <- NULL # YOUR CODE HERE (434 and 48)

se.even
se.lopsided

**Q4f.** Which split is more precise, and what does that suggest about how to divide people between treatment and control?

**Your answer:** *(double-click here to type your answer)*

## Step 5: More data, less noise

Now use **all four seasons** (the full `wnba` dataset) instead of just 2023.

**Q5a.** Run `difference_in_means()` on the full data, with `'away'` as baseline and `'home'` as treatment. Save it as `dim.all`.

In [ ]:
dim.all <- NULL # YOUR CODE HERE
dim.all

table(wnba$home_away)

**Q5b.** Compare the estimate, standard error, and t value to the 2023-only results. What changed a lot and what barely changed?

**Your answer:** *(double-click here to type your answer)*

**Q5c.** The full data has about 4 times as many rows as 2023 (986 per group vs. 241). Using what you learned in Step 4, about how much should the SE have shrunk? Did it?

**Your answer:** *(double-click here to type your answer)*

**Q5d.** Now |t| is above 1.96. Does that mean we know that being at home *causes* teams to score more? Would collecting even more seasons fix that concern? (Think about bias vs. noise.)

**Your answer:** *(double-click here to type your answer)*

## Step 6: A different outcome: winning

The variable `win` is 1 if the team won and 0 if it lost. Because it's 0/1, the mean of `win` is the **proportion** of games won.

Run this two-way table (rows = home/away, columns = lost/won):

In [ ]:
table(wnba$home_away, wnba$win)

**Q6a.** Use `subset()` and `mean()` to find the share of home teams' games that they won.

In [ ]:
home.all <- NULL # YOUR CODE HERE
mean(home.all$win)

**Q6b.** Now estimate the effect of playing at home on winning with `difference_in_means()` on the full data.

In [ ]:
dim.win <- NULL # YOUR CODE HERE
dim.win

**Q6c.** Interpret the estimate in a sentence (hint: it's in *proportion* units, so think in percentage points). Why is the away teams' win rate exactly 1 minus the home teams' win rate?

**Your answer:** *(double-click here to type your answer)*

## Step 7: The big question, bias

Everything so far (SE, t-statistics, shuffling) was about **noise**. But noise isn't the only problem. Nobody randomly assigned teams to home or away, so could the gap be **biased**?

One way to check: if home/away were "as good as random," the *kinds* of teams playing at home should look like the *kinds* of teams playing away. Here's a two-way table of team by home/away:

In [ ]:
table(wnba$team, wnba$home_away)

**Q7a.** Does each team play about as many home games as away games? Why does that help rule out one source of bias (for example, strong teams getting more home games)?

**Your answer:** *(double-click here to type your answer)*

**Q7b.** Name two other things that might differ between home and away games and might affect points scored. For each, which direction would it push the estimate?

**Your answer:** *(double-click here to type your answer)*

**Q7c.** Summarize: which parts of today's activity addressed **noise** and which addressed **bias**? Could a smaller standard error ever fix bias?

**Your answer:** *(double-click here to type your answer)*

## Step 8: Wrap-up

**Q8.** Imagine a sports reporter asks, *"Is home-court advantage real in the WNBA?"* In 3-4 sentences, answer using (1) your estimate, (2) how much noise there is, and (3) whether you'd call it a causal effect.

**Your answer:** *(double-click here to type your answer)*